In [1]:
cd ../..

/app


/usr/local/lib/python3.9/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
%run run.py connect

2024-11-26 13:13:30,074::INFO::settings.py::Setting loglevel to INFO
2024-11-26 13:13:30,076::INFO::settings.py::Setting stores to {}
2024-11-26 13:13:30,077::INFO::settings.py::Setting database.misc.schema_prefix to 
2024-11-26 13:13:30,078::INFO::settings.py::Setting database.misc.create_tables to True
2024-11-26 13:13:30,079::INFO::settings.py::Setting enable_python_native_blobs to True
2024-11-26 13:13:30,080::INFO::settings.py::Setting database.host to 128.178.51.167:3309
2024-11-26 13:13:30,081::INFO::settings.py::Setting database.user to thomas
2024-11-26 13:13:30,083::INFO::settings.py::Setting database.password to thomas
2024-11-26 13:13:30,142::INFO::connection.py::Connected thomas@128.178.51.167:3309
2024-11-26 13:13:30,157::INFO::table.py::could not log event in table ~log


Connecting thomas@128.178.51.167:3309


2024-11-26 13:13:30,562::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,562::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,575::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,575::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,601::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,601::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,633::INFO::table.py::could not log event in table ~log
2024-11-26 13:13:30,633::INFO::table.py::could not log event in table ~log


In [59]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns


from vr4mice.schema import base_analysis, dlc
from vr4mice.analysis import utils, analysis, regression, plotting

In [4]:
def get_all_in_list(data_set_list, training_stage):
    print(training_stage)
    big_df = []
    
    for d in data_set_list:
        split_d = d["dataset"].split("_")
        print(split_d)
        
        offline_kinematics_df = dlc.OfflineKinematics().get_data(key=d, columns = ["head_center_x", "head_center_y", "heading_dir", "head_angle"])
        df = base_analysis.DataFrame().get_data(key=d, 
                                                        columns=[
                                                            'dataset', 'trial', 'aperture',
                                                            'trial_right_choice', 'trial_left_choice',
                                                            'velocity', 'velocity_x', 'velocity_y',
                                                            "reward",
                                                            'norm_y', "iti", "x", "y",
                                                            'trial_init_x', 'trial_init_y',
                                                            "trial_tortuosity", "trial_duration"
        ])
        df["trial_rewarded"] = base_analysis.DataFrame().get_rewarded(key=d)
        
        df = df.join(offline_kinematics_df)
        
        df["mouse_name"] = split_d[0]
        df["date"] = split_d[1]
        df["attempt"] = split_d[2]
        df["training_stage"] = training_stage
        
        big_df.append(df)
    
    big_df =  pd.concat(big_df).reset_index()
    big_df ["session_increment"] = np.array(big_df.groupby("dataset").ngroup()+1)
    big_df = big_df.infer_objects()
    
    return(big_df.reset_index(drop=True))

In [5]:
dual_occuder = [{"dataset": "Nightingale_2024-08-14_1"},
                {"dataset": "Nightingale_2024-08-13_1"},
                {"dataset": "Nightingale_2024-08-12_1"},
                {"dataset": "Nightingale_2024-08-11_1"},
                {"dataset": "Nightingale_2024-08-10_1"},
                {"dataset": "Lemming_2024-08-13_1"},
                {"dataset": "Lemming_2024-08-12_1"},
                {"dataset": "Lemming_2024-08-11_1"},
                {"dataset": "Lemming_2024-08-10_1"},
                {"dataset": "Jacana_2024-08-13_1"},
                {"dataset": "Jacana_2024-08-14_1"},
                {"dataset": "Jacana_2024-08-15_1"},
                {"dataset": "Jacana_2024-08-16_1"},
                {"dataset": "Jacana_2024-08-19_1"},
                {"dataset": "Kiwi_2024-08-10_2"},
                {"dataset": "Kiwi_2024-08-11_4"},
                {"dataset": "Kiwi_2024-08-12_2"},
                {"dataset": "Kiwi_2024-08-13_1"},
                {"dataset": "Kiwi_2024-08-14_1"},
                {"dataset": "Oribi_2024-08-16_1"},
                {"dataset": "Oribi_2024-08-19_1"},
                {"dataset": "Oribi_2024-08-20_1"},
                {"dataset": "Oribi_2024-08-21_1"},
                {"dataset": "Oribi_2024-08-22_1"},
                {"dataset": "Pheasant_2024-08-15_2"},
                {"dataset": "Pheasant_2024-08-16_1"},
                {"dataset": "Pheasant_2024-08-20_1"},
                {"dataset": "Pheasant_2024-08-21_1"},
               ]

In [6]:
#big_df = get_all_in_list(dual_occuder, training_stage="dual_occlder")
#big_df.to_pickle("notebooks/Paper_figures/dual_occluder.pickle")
big_df = pd.read_pickle("notebooks/Paper_figures/dual_occluder.pickle")
big_df = big_df [big_df.iti ==0.0]

In [7]:
big_df["norm_x"] = big_df.groupby(["dataset", "trial"], as_index=False)["x"].transform(
        lambda x: x - np.mean(x.iloc[:3])
    )


big_df["flip_one_side"] = big_df["trial_left_choice"].replace([0, 1], [1, -1])
columns = [
    "norm_y",
    "norm_x",
    "heading_dir",
    "head_angle",
    "trial_tortuosity",
    "trial_duration",
    "x",
    "y",
    "aperture",
    "velocity",
    "velocity_x",
    "velocity_y",
    "trial_rewarded",
    "norm_y",
    "flip_one_side",
]
j_shaped = analysis.get_jshaped_trials(big_df)

#j_shaped = j_shaped[j_shaped["trial_rewarded"]==1]

n_samples = 500
interpolated_j_shaped = utils.interpolate(
    j_shaped, n_points=n_samples, value_columns=["trial_left_choice"] + columns
)
interpolated_j_shaped["trial_step"] = interpolated_j_shaped.groupby(
    ["dataset", "trial"], as_index=False).trial.cumcount()


interpolated_j_shaped["trial_length"] = interpolated_j_shaped["trial_step"] / n_samples
interpolated_j_shaped["head_angle_sin"] = np.sin(np.deg2rad(interpolated_j_shaped.head_angle))
interpolated_j_shaped["head_angle_cos"] = np.cos(np.deg2rad(interpolated_j_shaped.head_angle))

interpolated_j_shaped["heading_dir_sin"] = np.sin(np.deg2rad(interpolated_j_shaped.heading_dir))
interpolated_j_shaped["heading_dir_cos"] = np.cos(np.deg2rad(interpolated_j_shaped.heading_dir))

interpolated_j_shaped["velocity_x_fliped"] = (
    interpolated_j_shaped["velocity_x"] * interpolated_j_shaped["flip_one_side"]
)

In [35]:
model_labels = [
    "norm_x",
    "norm_y",
    "velocity_x",
    "velocity_y",
    "heading_dir_sin",
    "heading_dir_cos",
    "head_angle_sin",
    "head_angle_cos",
    "trial_tortuosity",
    "trial_duration",
    "aperture",
    "trial_rewarded",
    "trial_length",
]

In [38]:
interpolated_j_shaped["aperture"] = interpolated_j_shaped["aperture"].astype(float)

df_model, coef = regression.predict_decision(
    df=interpolated_j_shaped, label=model_labels, per_mouse=True)

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
group = df_model[(df_model.dataset == df_model.dataset.unique()[15])]

trials = [94 , 15,  66, 170, 224, 195,  56, 203,  88, 239, 113,  91, 186, 248, 109, 164,
          188,  60, 229, 182, 156, 197,  52,  45, 110, 156, 190,  22, 210,  24,  51,  78, 
          239,  97,  24, 176, 168, 184, 123, 156]

group = group [group.trial.isin(np.array(trials))]
sns.lineplot(
        data=group,
        x="trial_length",
        y="proba_left",
        hue="trial_left_choice",
        errorbar=None,
        estimator=None,
        units="trial",
        palette= plotting.colors_choice,
        sort=False, alpha=0.5, ax=ax[0]
    )
ax[0].set_ylabel("Proba(Left)")
ax[0].set_xlabel("Trial progression")

sns.lineplot(
        data=df_model,
        x="trial_length",
        y="proba_left",
        hue="trial_left_choice",
        style="aperture",
        palette= plotting.colors_choice,
        sort=False, alpha=0.5, ax=ax[1]
    )
ax[1].set_ylabel("Proba(Left)")
ax[1].set_xlabel("Trial progression")

In [101]:
trials = trials_prob.trial [(trials_prob.proba_left > 0.4) & (trials_prob.proba_left > 0.6)]

In [69]:
trials_prob.isin(np.array(trials))

,time,trial_left_choice,norm_y,norm_x,heading_dir,head_angle,trial_tortuosity,trial_duration,x,y,...,trial,trial_step,trial_length,head_angle_sin,head_angle_cos,heading_dir_sin,heading_dir_cos,velocity_x_fliped,accuracy,proba_left
0,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
500,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
1000,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1500,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2000,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121000,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
121500,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
122000,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
122500,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [70]:
group = group [trials_prob.isin(np.array(trials))]

In [71]:
group

,time,trial_left_choice,norm_y,norm_x,heading_dir,head_angle,trial_tortuosity,trial_duration,x,y,...,trial,trial_step,trial_length,head_angle_sin,head_angle_cos,heading_dir_sin,heading_dir_cos,velocity_x_fliped,accuracy,proba_left
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123496,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
